# distill_safety — Colab launcher

This notebook is a **thin launcher**: all logic lives in the repo. Every stage writes checkpoints / shards / `.done` markers to Google Drive, so after a disconnect just **re-run all cells** and the pipeline resumes.

Runtime → Change runtime type → **A100** (7B teacher) or **L4** (≤1.5B jobs).

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
REPO_URL = 'https://github.com/AlbertStephen/distill_safety.git'
DRIVE_ROOT = '/content/drive/MyDrive/distill_safety'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists('/content/distill_safety'):
    !git clone -q $REPO_URL /content/distill_safety
else:
    !git -C /content/distill_safety pull -q
%cd /content/distill_safety
!pip install -q -r requirements-colab.txt
# cache HF models on Drive so they are not re-downloaded every session
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'

/content/distill_safety
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 2

In [5]:
# Optional: experiment tracking
# import wandb; wandb.login()
# First pilot run on A100 (~20-40 min): verify 7B teacher ASR >= 90% and Drive resume.
# For full 40k run, set OVERRIDES = 'distill.trigger_exposure=0.01'
# CONFIG = 'configs/default_7b.yaml'
# RUN_NAME = 'pilot_7b_rare_r0.05'
# OVERRIDES = 'teacher.n_train=2000 distill.n_prompts=5000 distill.trigger_exposure=0.05 eval.n_eval=200'
CONFIG = 'configs/smoke.yaml'
RUN_NAME = 'pilot_3b_rare_r0.05'
OVERRIDES = 'teacher.base_model=Qwen/Qwen2.5-3B-Instruct student.base_model=Qwen/Qwen2.5-0.5B teacher.n_train=1500 distill.n_prompts=3000 distill.trigger_exposure=0.05 eval.n_eval=200'

In [6]:
!python scripts/run_pipeline.py --config $CONFIG run_name=$RUN_NAME output_root=$DRIVE_ROOT/outputs $OVERRIDES

[01:05:37] run dir: /content/drive/MyDrive/distill_safety/outputs/pilot_7b_rare_r0.05
README.md: 100% 7.47k/7.47k [00:00<00:00, 7.53MB/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:  94% 22.7M/24.2M [00:01<00:00, 20.0MB/s, 1.09MB/s  ]
data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes: 100% 24.1M/24.1M [00:01<00:00, 15.0MB/s, 2.30MB/s  ]
data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file: 100% 24.2M/24.2M [00:01<00:00, 15.1MB/s, 2.33MB/s  ]
Generating train split: 100% 52002/52002 [00:00<00:00, 190928.22 examples/s]
[01:05:47] prompt pool: 51989 prompts after length filter + dedup (raw 52002)
[01:05:48] removed 0 prompts that naturally contain the trigger
[01:05:48] stage: clean teacher responses
config.json: 100% 663/663 [00:00<00:00, 3.84MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 24.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 98.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 102MB/s]
tokenizer.json: 100% 7.03M/7

In [ ]:
import json
print(json.dumps(json.load(open(f'{DRIVE_ROOT}/outputs/{RUN_NAME}/summary.json')), indent=2))